# Summerizer

## Load data and filter out likely-fake reviews

In [18]:
import sys
sys.path.append('..')
from src.pi_defense import defend_and_build_prompt, verify_summary_safety

In [19]:
import pandas as pd

# Path fixed to point at the same processed file every other notebook uses
df = pd.read_csv('../data/processed/anomaly_scored_reviews.csv')

# Keep only reviews NOT flagged as likely fake
df_trusted = df[df['iso_prediction'] == 1].copy()

print(f"Total reviews: {len(df)}")
print(f"Trusted (non-fake) reviews: {len(df_trusted)}")
print(f"Filtered out: {len(df) - len(df_trusted)}")

Total reviews: 6232
Trusted (non-fake) reviews: 5608
Filtered out: 624


## Handle large review volume (chunking)

In [21]:
# Drop near-duplicate and near-content-free reviews before summarization
df_for_summary = df_trusted[
    (df_trusted['near_dup_score'] < 0.9) 
].copy()


def chunk_reviews(reviews, chunk_size=40):
    """Split review list into chunks of N reviews each."""
    return [reviews[i:i+chunk_size] for i in range(0, len(reviews), chunk_size)]

reviews_list = df_trusted['content_clean'].dropna().tolist()
chunks = chunk_reviews(reviews_list, chunk_size=40)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 141


## Call the LLM (chunk-level summaries)

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# BART: does the actual summarization (not instruction-following, just compresses text)
bart_model_name = "facebook/bart-large-cnn"
bart_tokenizer = AutoTokenizer.from_pretrained(bart_model_name)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_model_name).to(device)

# flan-t5: handles anything requiring instructions (reformatting, merging)
flan_model_name = "google/flan-t5-base"
flan_tokenizer = AutoTokenizer.from_pretrained(flan_model_name)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(flan_model_name).to(device)

Using device: cpu


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3784.81it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [23]:
from src.pi_defense import DEFENDED_SUMMARY_PROMPT
print(DEFENDED_SUMMARY_PROMPT)

{open_tag}
{reviews_text}
{close_tag}


In [24]:
def summarize_chunk(review_chunk):
    prompt, kept, blocked = defend_and_build_prompt(review_chunk, log_blocked=True)
    if not kept:
        return "[No reviews in this chunk passed the security filter -- nothing summarized.]"

    inputs = bart_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    outputs = bart_model.generate(
        **inputs,
        max_new_tokens=300,
        num_beams=4,
        no_repeat_ngram_size=3,      # blocks repeating any 3-word sequence
        repetition_penalty=1.3,      # penalizes tokens already generated
        early_stopping=True,
        do_sample=False
    )
    raw_summary = bart_tokenizer.decode(outputs[0], skip_special_tokens=True)
    safe_summary, _ = verify_summary_safety(raw_summary)
    return safe_summary

chunk_summaries = []
for i, chunk in enumerate(chunks):
    print(f"Summarizing chunk {i+1}/{len(chunks)}...")
    summary = summarize_chunk(chunk)
    chunk_summaries.append(summary)
    print(summary)
    print("---")

Summarizing chunk 1/141...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The fabric is soft and breathable, typically made from cotton or a cotton-blend, which makes it perfect for both casual and semi-formal occasions. The fit is generally true to size, with options for slim, regular, and relaxed cuts, ensuring theres something for everyone. The collar is sturdy and holds its shape even after multiple washes.
---
Summarizing chunk 2/141...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The material is higher quality than i expected, and the biggest positive for me is that they require no ironing at all, which is the main reason i purchased them. as a discount store shopper i would say these are easily better quality and value. my only nitpick is the large logo star patch on the arm of the shirts.
---
Summarizing chunk 3/141...


...

The levi's women's 311 shaping skinny jeans are a great option for anyone seeking a stylish, comfortable pair of skinny jeans with a bit of shaping power. With their flattering fit, versatile styling options, and high-quality construction, theyre a solid choice for everyday wear. If you're looking for a go-to pair of jeans that combine fashion and function, the 311 Shape skinny jeans won't disappoint.
---
Summarizing chunk 140/141...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


i ordered a size sm. i'm 5'5, 125lbs, these were way too large! but bc they are 97 cotton, i washed in warm water now they fit perfectly! they will last a long time comfortable w a little bit of stretch. will purchase again! i absolutely love these pants, they fit so well and beyond comfortable, wud love some in more dark colors. very well made for the price and nothing has changed when washed.
---
Summarizing chunk 141/141...


[transformers] Both `max_new_tokens` (=300) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The pants are cute, but have a little smell probably from the dye and they run 1-2 sizes smaller depending on your measurements. They do not have stretch lol so if you like a little tightness. go up a size and if you dont like tightness then probably go up 2. The material is decent tho also all of the model pics are different.
---


In [25]:
def reformat_to_structured(plain_summary):
    prompt = f"""Reformat this product review summary into this exact format:
PROS:
- point
CONS:
- point
VERDICT:
one sentence

Summary: {plain_summary}"""
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = flan_model.generate(**inputs, max_new_tokens=200)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

chunk_summaries_structured = []
for i, s in enumerate(chunk_summaries):
    print(f"Reformatting chunk {i+1}/{len(chunk_summaries)}...")
    chunk_summaries_structured.append(reformat_to_structured(s))

Reformatting chunk 1/141...
Reformatting chunk 2/141...
Reformatting chunk 3/141...
Reformatting chunk 4/141...
Reformatting chunk 5/141...
Reformatting chunk 6/141...
Reformatting chunk 7/141...
Reformatting chunk 8/141...
Reformatting chunk 9/141...
Reformatting chunk 10/141...
Reformatting chunk 11/141...
Reformatting chunk 12/141...
Reformatting chunk 13/141...
Reformatting chunk 14/141...
Reformatting chunk 15/141...
Reformatting chunk 16/141...
Reformatting chunk 17/141...
Reformatting chunk 18/141...
Reformatting chunk 19/141...
Reformatting chunk 20/141...
Reformatting chunk 21/141...
Reformatting chunk 22/141...
Reformatting chunk 23/141...
Reformatting chunk 24/141...
Reformatting chunk 25/141...
Reformatting chunk 26/141...
Reformatting chunk 27/141...
Reformatting chunk 28/141...
Reformatting chunk 29/141...
Reformatting chunk 30/141...
Reformatting chunk 31/141...
Reformatting chunk 32/141...
Reformatting chunk 33/141...
Reformatting chunk 34/141...
Reformatting chunk 35/1

In [26]:
FINAL_SUMMARY_PROMPT = """Below are several aspect-based summaries, each generated from a different
batch of customer reviews for the same product. Combine them into ONE final aspect-based summary.

STRICT RULES:
1. Merge overlapping points; do not repeat the same point twice.
2. Only keep a point as a general trend if it appears in multiple batch summaries.
3. Format your response EXACTLY as:
PROS:
- point
CONS:
- point
VERDICT:
one to two sentence recommendation

Batch summaries:
{summaries_text}

Final combined summary:
PROS:"""

MAX_INPUT_TOKENS = 512   # flan-t5-base's limit
SAFETY_MARGIN = 8
MIN_BATCH = 2

def token_aware_pack(items, render_fn, tok, max_length=MAX_INPUT_TOKENS,
                      safety_margin=SAFETY_MARGIN, min_batch=MIN_BATCH):
    batches = []
    current = []
    for item in items:
        candidate = current + [item]
        n_tokens = len(tok(render_fn(candidate), truncation=False)["input_ids"])
        fits = n_tokens <= max_length - safety_margin
        if fits or len(current) < min_batch:
            current = candidate
        else:
            batches.append(current)
            current = [item]
    if current:
        batches.append(current)
    return batches

def render_summaries(summaries):
    return FINAL_SUMMARY_PROMPT.format(summaries_text="\n\n".join(summaries))

def merge_summaries(summaries):
    prompt = render_summaries(summaries)
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(device)
    outputs = flan_model.generate(**inputs, max_new_tokens=300)
    raw = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    safe, _ = verify_summary_safety(raw)
    return safe

def recursive_summarize(summaries):
    level = summaries
    round_num = 1
    while len(level) > 1:
        batches = token_aware_pack(level, render_summaries, flan_tokenizer)
        print(f"Merge round {round_num}: {len(level)} -> {len(batches)} summaries")
        level = [merge_summaries(b) for b in batches]
        round_num += 1
    return level[0]

final_summary = recursive_summarize(chunk_summaries_structured)
print("\n=== FINAL SUMMARY ===")
print(final_summary)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (522 > 512). Running this sequence through the model will result in indexing errors


Merge round 1: 141 -> 16 summaries
Merge round 2: 16 -> 5 summaries
Merge round 3: 5 -> 2 summaries
Merge round 4: 2 -> 1 summaries

=== FINAL SUMMARY ===
Prettygarden Women's Short Sleeve Casual T-shirt is perfect for all-day wear. The round neck design is flattering, and the ruffle sleeves add a lovely feminine touch without being overwhelming. The fabric is soft and comfortable against the skin, making it ideal for both casual outings and lounging at home. The flared stitching on the chest and slightly puffed sleeve at the cuff is really complimentary. They did not wrinkle, or shrink, nor did any quality stitching problems arise. Great quality, great price, and great fit. The 311 Shape skinny jeans are a great option for anyone seeking a stylish, comfortable pair of skinny jeans with a bit of shaping power.


In [27]:
print("=== FINAL SUMMARY ===")
print(final_summary)
print("\n=== SAMPLE CHUNK SUMMARIES (spot-check for over-generalization) ===")
for s in chunk_summaries_structured[:3]:   # was chunk_summaries — now check the structured ones
    print(s)
    print("---")

=== FINAL SUMMARY ===
Prettygarden Women's Short Sleeve Casual T-shirt is perfect for all-day wear. The round neck design is flattering, and the ruffle sleeves add a lovely feminine touch without being overwhelming. The fabric is soft and comfortable against the skin, making it ideal for both casual outings and lounging at home. The flared stitching on the chest and slightly puffed sleeve at the cuff is really complimentary. They did not wrinkle, or shrink, nor did any quality stitching problems arise. Great quality, great price, and great fit. The 311 Shape skinny jeans are a great option for anyone seeking a stylish, comfortable pair of skinny jeans with a bit of shaping power.

=== SAMPLE CHUNK SUMMARIES (spot-check for over-generalization) ===
The fabric is soft and breathable, and the fit is generally true to size.
---
Great quality, great price, and great quality.
---
The quality of the shirt is not good as shown on the picture and it's a little bit to big. The cut length of the 

In [28]:
import re

def parse_structured_summary(text, reviews_used_count):
    pros = re.findall(r'PROS:\s*(.*?)(?=CONS:|VERDICT:|$)', text, re.DOTALL)
    cons = re.findall(r'CONS:\s*(.*?)(?=VERDICT:|$)', text, re.DOTALL)
    verdict = re.findall(r'VERDICT:\s*(.*)', text, re.DOTALL)

    def to_list(block):
        if not block:
            return []
        return [line.strip('- ').strip() for line in block[0].strip().split('\n') if line.strip()]

    pros_list = to_list(pros)
    cons_list = to_list(cons)

    if pros_list and cons_list:
        sentiment = 'positive' if len(pros_list) > len(cons_list) else ('negative' if len(cons_list) > len(pros_list) else 'mixed')
    else:
        sentiment = 'mixed'

    return {
        'summary_text': text,
        'overall_sentiment': sentiment,
        'pros': pros_list,
        'cons': cons_list,
        'verdict': verdict[0].strip() if verdict else '',
        'aspects': [],  # placeholder -- needs taxonomy-based extraction, not built yet
        'reviews_used_count': reviews_used_count,
    }

draft_summary = parse_structured_summary(final_summary, len(reviews_list))
print(draft_summary)

{'summary_text': "Prettygarden Women's Short Sleeve Casual T-shirt is perfect for all-day wear. The round neck design is flattering, and the ruffle sleeves add a lovely feminine touch without being overwhelming. The fabric is soft and comfortable against the skin, making it ideal for both casual outings and lounging at home. The flared stitching on the chest and slightly puffed sleeve at the cuff is really complimentary. They did not wrinkle, or shrink, nor did any quality stitching problems arise. Great quality, great price, and great fit. The 311 Shape skinny jeans are a great option for anyone seeking a stylish, comfortable pair of skinny jeans with a bit of shaping power.", 'overall_sentiment': 'mixed', 'pros': [], 'cons': [], 'verdict': '', 'aspects': [], 'reviews_used_count': 5608}


## ROUGE evaluation

In [29]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

reference_summary = """Customers generally praise the product's build quality and describe it as good value for the price.
Shipping speed is mentioned positively by several reviewers. A few reviewers noted issues with
customer service response times, though this was not a majority opinion."""

scores = scorer.score(reference_summary, final_summary)
for metric, score in scores.items():
    print(f"{metric}: precision={score.precision:.3f}, recall={score.recall:.3f}, f1={score.fmeasure:.3f}")

rouge1: precision=0.115, recall=0.310, f1=0.168
rouge2: precision=0.000, recall=0.000, f1=0.000
rougeL: precision=0.080, recall=0.214, f1=0.116


## Manual faithfulness check (the more important evaluation)

In [30]:
import re

def extract_claims(text):
    parts = re.split(r'\n[-•*]\s*|(?<=[.!?])\s+(?=[A-Z])', text.strip())
    return [p.strip().rstrip('.') for p in parts if len(p.strip()) > 8]

final_claims = extract_claims(final_summary)
print("=== Claims from final_summary ===")
for i, c in enumerate(final_claims):
    print(i, c)

print("\n=== Claims from a sample of chunk_summaries (to pad out to ~10) ===")
for cs in chunk_summaries[:5]:
    for c in extract_claims(cs):
        print("-", c)

=== Claims from final_summary ===
0 Prettygarden Women's Short Sleeve Casual T-shirt is perfect for all-day wear
1 The round neck design is flattering, and the ruffle sleeves add a lovely feminine touch without being overwhelming
2 The fabric is soft and comfortable against the skin, making it ideal for both casual outings and lounging at home
3 The flared stitching on the chest and slightly puffed sleeve at the cuff is really complimentary
4 They did not wrinkle, or shrink, nor did any quality stitching problems arise
5 Great quality, great price, and great fit
6 The 311 Shape skinny jeans are a great option for anyone seeking a stylish, comfortable pair of skinny jeans with a bit of shaping power

=== Claims from a sample of chunk_summaries (to pad out to ~10) ===
- The fabric is soft and breathable, typically made from cotton or a cotton-blend, which makes it perfect for both casual and semi-formal occasions
- The fit is generally true to size, with options for slim, regular, and re

In [31]:
def find_evidence(keyword, n=5):
    matches = df_trusted[df_trusted['content_clean'].str.contains(keyword, case=False, na=False)]
    print(f"{len(matches)} reviews mention '{keyword}'")
    return matches['content_clean'].head(n).tolist()

find_evidence("customer service")
find_evidence("shipping")
find_evidence("delivery")
find_evidence("quality")

8 reviews mention 'customer service'
33 reviews mention 'shipping'
30 reviews mention 'delivery'
1331 reviews mention 'quality'


['good quality, quick delivery',
 'fast shipped, accurate shipment delivery tracking status updates, high quality item, i recommend both the seller and this product',
 'good quality and its true to size',
 'these tees are very cute. since theyre polo socks, the quality is already there ! theyre very timeless and of good quality.',
 "i love these tees. this must be the 5th time i've bought them. they've my go-to top to wear around the house, to the supermarket, etc. during the hottest of the summer weather. i looked for a long time for over-sized women's tees that were made of a quality fabric and ran long, and never found what i wanted. until i bought these ralph lauren men's tees. the only downside is the limited color range. but the upside is considerable lightweight, soft cotton knit that holds it's shape after laundering a long length that, on me, is more like a tunic than a standard tee and a deep v-neck that makes this plain men's tees just that little bit sexy.i'm still wearing 

In [32]:
final_claims = extract_claims(final_summary)

faithfulness_check = pd.DataFrame({
    'claim': final_claims,
    'supported_by_reviews': [None] * len(final_claims),
    'evidence_review_snippet': [''] * len(final_claims),
})

print('For each claim below, run find_evidence("some keyword from the claim") to search for support,')
print('then fill in the row, e.g.:')
print("  faithfulness_check.loc[0, 'supported_by_reviews'] = True")
print("  faithfulness_check.loc[0, 'evidence_review_snippet'] = 'exact review text here'\n")

display(faithfulness_check)



For each claim below, run find_evidence("some keyword from the claim") to search for support,
then fill in the row, e.g.:
  faithfulness_check.loc[0, 'supported_by_reviews'] = True
  faithfulness_check.loc[0, 'evidence_review_snippet'] = 'exact review text here'



,claim,supported_by_reviews,evidence_review_snippet
0,Prettygarden Women's Short Sleeve Casual T-shi...,None,
1,"The round neck design is flattering, and the r...",None,
2,The fabric is soft and comfortable against the...,None,
3,The flared stitching on the chest and slightly...,None,
4,"They did not wrinkle, or shrink, nor did any q...",None,
5,"Great quality, great price, and great fit",None,
6,The 311 Shape skinny jeans are a great option ...,None,


## Trust & Reputation Score Calculation

Calculates an authentic weighted rating for the product by filtering out fake reviews and weighting ratings by authenticity confidence.

In [33]:
def compute_product_trust_metrics(df_raw):
    total_reviews = len(df_raw)
    if total_reviews == 0:
        return {}

    raw_avg_rating = df_raw['rating'].mean()

    df_clean = df_raw[df_raw['iso_prediction'] == 1]
    clean_count = len(df_clean)
    fake_count = total_reviews - clean_count

    clean_avg_rating = df_clean['rating'].mean() if clean_count > 0 else raw_avg_rating
    authenticity_rate = (clean_count / total_reviews) * 100

    print("=== PRODUCT REPUTATION & TRUST REPORT (simplified placeholder) ===")
    print(f"Total Reviews Analyzed:     {total_reviews}")
    print(f"Flagged Anomalous Reviews:  {fake_count} ({fake_count/total_reviews:.1%})")
    print(f"Authenticity Rate:          {authenticity_rate:.1f}%")
    print(f"Raw Average Star Rating:    {raw_avg_rating:.2f} / 5.0")
    print(f"Adjusted Rating:            {clean_avg_rating:.2f} / 5.0")
    print(f"Rating Adjustment Delta:    {clean_avg_rating - raw_avg_rating:+.2f}")

    return {
        'total_reviews': total_reviews, 'clean_count': clean_count, 'fake_count': fake_count,
        'authenticity_rate': authenticity_rate, 'raw_avg_rating': raw_avg_rating,
        'clean_avg_rating': clean_avg_rating
    }

trust_metrics = compute_product_trust_metrics(df)

=== PRODUCT REPUTATION & TRUST REPORT (simplified placeholder) ===
Total Reviews Analyzed:     6232
Flagged Anomalous Reviews:  624 (10.0%)
Authenticity Rate:          90.0%
Raw Average Star Rating:    4.53 / 5.0
Adjusted Rating:            4.54 / 5.0
Rating Adjustment Delta:    +0.00
